In [ ]:
"""
b2c_sales_report_generator.py

Automates the generation of a B2C sales and refund report from a monthly CSV export.
The script extracts the report date from the file name, processes shipment and refund
data, groups by SKU and location, and exports formatted Excel sheets.

Author: Mala Dhangar
"""

import os
import pandas as pd
from datetime import datetime
from calendar import monthrange


def extract_last_date_from_filename(filename: str) -> str:
    try:
        month_year = filename.split('.')[0]
        month_str, year_str = month_year.split()
        month = datetime.strptime(month_str[:3], "%b").month
        year = int(year_str)
        last_day = monthrange(year, month)[1]
        return datetime(year, month, last_day).strftime("%d-%m-%Y")
    except Exception as e:
        raise ValueError(f"Filename '{filename}' does not match expected 'Mon YYYY.csv' format.") from e


def load_and_tag_data(file_path: str, report_date: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    df.insert(0, 'Date', report_date)
    return df


def process_transaction_data(df: pd.DataFrame, transaction_type: str) -> pd.DataFrame:
    df = df[df['Transaction Type'] == transaction_type]

    required_cols = ['Date', 'Seller Gstin', 'Ship To State', 'Sku', 'Fulfillment Channel',
                     'Quantity', 'Tax Exclusive Gross', 'Cgst Tax', 'Sgst Tax', 'Igst Tax', 'Invoice Amount']
    df = df[required_cols]

    numeric_cols = required_cols[5:]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    grouped = df.groupby(['Date', 'Seller Gstin', 'Ship To State', 'Sku', 'Fulfillment Channel'], as_index=False).sum()
    grouped['sort_key'] = grouped.apply(lambda row: (row['Ship To State'], 0 if row['Fulfillment Channel'] == 'AFN' else 1), axis=1)
    return grouped.sort_values(by='sort_key').drop(columns=['sort_key'])


def write_to_excel(shipment_df: pd.DataFrame, refund_df: pd.DataFrame, output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        for sheet, df in zip(['Shipment', 'Refund'], [shipment_df, refund_df]):
            df.to_excel(writer, sheet_name=sheet, index=False)
            worksheet = writer.sheets[sheet]
            workbook = writer.book
            worksheet.add_table(0, 0, df.shape[0], df.shape[1] - 1, {
                'columns': [{'header': col} for col in df.columns],
                'style': 'Table Style Medium 9'
            })


def main():
    folder_path = 'C:/Users/malad/Jitin'
    output_folder = 'C:/Users/malad/Jitin/outputs'
    
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in the specified folder.")
    
    input_file = os.path.join(folder_path, csv_files[0])
    report_date = extract_last_date_from_filename(os.path.basename(input_file))
    
    full_data = load_and_tag_data(input_file, report_date)
    shipment_report = process_transaction_data(full_data, 'Shipment')
    refund_report = process_transaction_data(full_data, 'Refund')

    output_path = os.path.join(output_folder, 'B2C_Sales_Report_Output.xlsx')
    write_to_excel(shipment_report, refund_report, output_path)

    print(f"✅ Report generated successfully: {output_path}")


if __name__ == "__main__":
    main()
